1. we load the document
2. then we will preprocess or clean the text
3. chunk the text
4. then we would do sparse search
5. then we would do dense search thus we will also implement database
6. we will do hybrid search
7. now we give detched data and query to chatbot to generate answer

In [1]:
! pip install pypdf spacy sentence-transformers scikit-learn rank-bm25 chromadb openai numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

we load all the dependencies

In [2]:
from google.colab import userdata
import numpy as np
import spacy
import os
import pypdf as PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import chromadb
from openai import OpenAI

now setup all the models

In [3]:
nlp=spacy.load("en_core_web_sm")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder("ms-marco-MiniLM-L-6-v2")
api_key = userdata.get("raj_api_key")
client_openai = OpenAI(api_key=api_key)
client= chromadb.PersistentClient("/content/sample_data/persistant_storagee")
collection = client.get_or_create_collection("raj_collection")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

now we load our pdf

In [4]:
def load_multiple_pdfs(folder_path):
  documents=[]

  for file in os.listdir(folder_path):
    if file.endswith(".pdf"):
      file_path=os.path.join(folder_path, file)
      reader = PdfReader.PdfReader(file_path)
      text=" "
      for pages in reader.pages:
        text+=pages.extract_text() + '\n'
      documents.append({
              "text": text,
              "source": file
          })
    elif file.endswith(".txt"):
      file_path=os.path.join(folder_path, file)
      with open(file_path, "r") as f:
        text=f.read()
        documents.append({
              "text": text,
              "source": file
          })



  return documents

now we preprocess the text extracted

In [5]:
def preprocess_text(text):
  tokens=[]
  doc=nlp(text.lower())
  for token in doc:
    if not token.is_stop and not token.is_punct:
      tokens.append(token.lemma_)
  return " ".join(tokens)


now we chunk our text

In [6]:
def chunking(text, threshold=0.3):
  sentences = [sent.text.strip() for sent in nlp(text).sents]
  embedding = embed_model.encode(sentences)

  chunks=[]
  current_chunk=[sentences[0]]
  for i in range(1, len(sentences)):
    sim=cosine_similarity([embedding[i-1]],[embedding[i]])[0][0]
    if sim> threshold:
      current_chunk.append(sentences[i])
    else:
      chunks.append(" ".join(current_chunk))
      current_chunk=[sentences[i]]
  chunks.append(" ".join(current_chunk))
  return chunks




sparse search bm25 algorithm and embedding for dense search

In [7]:
#indexing my data
def index(documents):
  metadata=[]
  sim_chunks=[]

  for doc in documents:
    chunks_from_doc = chunking(doc['text'])
    doc['sim_chunk']= chunks_from_doc

    for chunk in chunks_from_doc:
      sim_chunks.append(chunk)
      metadata.append({"source": doc["source"]})

  #bm25
  processed_chunks = [preprocess_text(c) for c in sim_chunks]
  tokenized_chunks=[doc.split(" ") for doc in processed_chunks]
  bm25=BM25Okapi(tokenized_chunks)

  #dense embedding
  chunk_embed=embed_model.encode(sim_chunks)
  collection.add(
      documents=sim_chunks,
      embeddings=chunk_embed.tolist(),
      metadatas=metadata,
      ids=[f"id{i}" for i in range(len(sim_chunks))]
  )

  return bm25, tokenized_chunks, sim_chunks

Hybrid retrival

In [8]:
def hybrid_retrival(query, bm25, tokenized_chunks, sim_chunks, top_k=5):
  #bm25
  processed_query = preprocess_text(query)
  tokenized_query = processed_query.split(" ")

  bm25_scores = bm25.get_scores(tokenized_query)

  #dense
  query_embed=embed_model.encode([query])[0]
  doc_embed=embed_model.encode(sim_chunks)

  dense_scores = cosine_similarity([query_embed], doc_embed)[0]

  #normalizing the scores
  bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) - np.min(bm25_scores) + 1e-8)
  dense_scores = (dense_scores - np.min(dense_scores)) / (np.max(dense_scores) - np.min(dense_scores) + 1e-8)

  hybrid_scores = 0.5 * bm25_scores + 0.5 * dense_scores

  top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

  return [sim_chunks[i] for i in top_indices]


chatbot

In [9]:
def llm(query, docs):
  context="\n\n".join(docs)
  message = [{
          "role":"system",
          "content":"""you are helpful assistant, expert in clarifying doubts related to health insaurance.
           You make sure to give relevent details after reading all the terms and conditions of the insaurance,
           If the query asked is an universal truth then answer it using your on trained dataset.
           else:
            the data about query asked is not present in the uploaded document then simply say 'I don't know'.
           """
                },
          {
            "role":"user",
            "content":f"Context:\n{context}\n\nQuestion: {query}"
                }]
  response=client_openai.chat.completions.create(
      model="gpt-4.1-mini",
      messages= message,
      temperature=0.1
  )

  answer = response.choices[0].message.content
  message.append({"role":"assistant", "content":answer})
  return answer

main function to call my rag model

In [10]:
if __name__ == "__main__":
    folder_path = "/content/sample_data/input_documents"

    print("Loading PDFs...")
    documents = load_multiple_pdfs(folder_path)

    print("Building index...")
    bm25, tokenized_chunks, sim_chunks = index(documents)

    print("RAG Ready! Ask questions")

    while True:
        query = input("\nYou: ")

        if query.lower() == "exit" or query.lower() == "quit":
            break

        retrieved = hybrid_retrival(query, bm25, tokenized_chunks, sim_chunks)

        answer = llm(query, retrieved)

        print("\nAI:", answer)

Loading PDFs...
Building index...
RAG Ready! Ask questions

You: how much premium do i need to pay if my employee grade is e1

AI: If your employee grade is E1, the premium you need to pay for the Medical Insurance Base Plan is INR 10,000.

You: exit
